# Module 3 — Exam Score Predictor (XGBoost Regression)
### AI-Powered Student LMS | Phase 3

**Goal:** Predict a student's final semester grade using mid-semester features  
**Target:** `Curricular units 2nd sem (grade)` — final semester grade (0–20 scale)  
**Model:** XGBoost Regressor + Linear Regression (baseline comparison)  
**Metrics:** RMSE, MAE, R² Score  

---

In [ ]:
# ── CELL 1 ── Imports & Paths ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from xgboost                 import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
import shap

BASE   = r'C:\Users\DELL\Desktop\AI_Student_LMS'
MODELS = os.path.join(BASE, 'models')
ASSETS = os.path.join(BASE, 'assets')

print('✅ All imports done!')

In [ ]:
# ── CELL 2 ── Load Data & Define Target ───────────────────────
df = pd.read_csv(os.path.join(BASE, 'data', 'raw', 'data.csv'), sep=';')
print(f'Dataset loaded: {df.shape}')

# Target: 2nd semester grade (final exam performance)
TARGET = 'Curricular units 2nd sem (grade)'

# Remove students with 0 grade (not enrolled / no grade recorded)
df = df[df[TARGET] > 0].reset_index(drop=True)
print(f'After removing 0-grade rows: {df.shape}')

print(f'\nTarget column stats:')
print(df[TARGET].describe().round(2))

# Feature columns — use 1st sem data to predict 2nd sem grade
# (realistic: teacher has access to 1st sem results mid-year)
FEATURE_COLS = [
    'Age at enrollment',
    'Admission grade',
    'Previous qualification (grade)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Scholarship holder',
    'Tuition fees up to date',
    'Debtor',
    'Gender',
    'Displaced',
]

# Keep only columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'\nFeatures used: {len(FEATURE_COLS)}')

X = df[FEATURE_COLS]
y = df[TARGET]
print(f'X shape: {X.shape} | y range: {y.min():.1f} – {y.max():.1f}')

In [ ]:
# ── CELL 3 ── Train-Test Split & Scale ────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_train_df = pd.DataFrame(X_train_sc, columns=FEATURE_COLS)
X_test_df  = pd.DataFrame(X_test_sc,  columns=FEATURE_COLS)

print(f'Train: {X_train_df.shape} | Test: {X_test_df.shape}')
print('✅ Data split and scaled!')

In [ ]:
# ── CELL 4 ── Baseline: Linear Regression ─────────────────────
lr = LinearRegression()
lr.fit(X_train_df, y_train)
lr_preds = lr.predict(X_test_df)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_r2   = r2_score(y_test, lr_preds)

print('=== Baseline: Linear Regression ===')
print(f'RMSE : {lr_rmse:.4f}')
print(f'MAE  : {lr_mae:.4f}')
print(f'R²   : {lr_r2:.4f}')

In [ ]:
# ── CELL 5 ── XGBoost Regressor ───────────────────────────────
xgb_reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_reg.fit(X_train_df, y_train)
xgb_preds = xgb_reg.predict(X_test_df)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae  = mean_absolute_error(y_test, xgb_preds)
xgb_r2   = r2_score(y_test, xgb_preds)

print('=== XGBoost Regressor ===')
print(f'RMSE : {xgb_rmse:.4f}')
print(f'MAE  : {xgb_mae:.4f}')
print(f'R²   : {xgb_r2:.4f}')

# Save model
joblib.dump(xgb_reg,     os.path.join(MODELS, 'score_predictor.pkl'))
joblib.dump(scaler,      os.path.join(MODELS, 'score_scaler.pkl'))
joblib.dump(FEATURE_COLS,os.path.join(MODELS, 'score_features.pkl'))
print('\n✅ XGBoost score predictor saved!')

In [ ]:
# ── CELL 6 ── PLOT 1: Actual vs Predicted ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title, color in zip(
    axes,
    [lr_preds, xgb_preds],
    ['Linear Regression (Baseline)', 'XGBoost Regressor'],
    ['#95A5A6', '#378ADD']
):
    ax.scatter(y_test, preds, alpha=0.35, s=15, color=color)
    # Perfect prediction line
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect prediction')
    ax.set_xlabel('Actual Grade', fontsize=11)
    ax.set_ylabel('Predicted Grade', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs Predicted — Exam Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'score_01_actual_vs_predicted.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Actual vs Predicted plot saved!')

In [ ]:
# ── CELL 7 ── PLOT 2: Residual Distribution ───────────────────
xgb_residuals = y_test.values - xgb_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual scatter
axes[0].scatter(xgb_preds, xgb_residuals, alpha=0.35, s=15, color='#378ADD')
axes[0].axhline(0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Grade')
axes[0].set_ylabel('Residual (Actual − Predicted)')
axes[0].set_title('Residual Plot — XGBoost', fontsize=12, fontweight='bold')

# Residual histogram
axes[1].hist(xgb_residuals, bins=40, color='#378ADD', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution — XGBoost', fontsize=12, fontweight='bold')

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'score_02_residuals.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Residual plots saved!')

In [ ]:
# ── CELL 8 ── PLOT 3: Model Comparison Bar Chart ──────────────
metrics = ['RMSE', 'MAE', 'R²']
lr_vals  = [lr_rmse,  lr_mae,  lr_r2]
xgb_vals = [xgb_rmse, xgb_mae, xgb_r2]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, lr_vals,  width, label='Linear Regression',
               color='#95A5A6', edgecolor='white')
bars2 = ax.bar(x + width/2, xgb_vals, width, label='XGBoost',
               color='#378ADD', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_title('Model Comparison — Regression Metrics', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'score_03_model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved!')

In [ ]:
# ── CELL 9 ── PLOT 4: Feature Importance ──────────────────────
feat_imp = pd.Series(xgb_reg.feature_importances_, index=FEATURE_COLS)
feat_imp = feat_imp.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#E74C3C' if v > feat_imp.quantile(0.75) else '#378ADD'
          for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors)
plt.title('Feature Importance — Score Predictor (XGBoost)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'score_04_feature_importance.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved!')

In [ ]:
# ── CELL 10 ── SHAP for Score Predictor ───────────────────────
print('Computing SHAP values for score predictor...')

explainer = shap.TreeExplainer(xgb_reg)
X_sample  = X_test_df.sample(200, random_state=42)
shap_vals = explainer.shap_values(X_sample)

# SHAP Summary Plot
plt.figure()
shap.summary_plot(
    shap_vals, X_sample,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Summary — Exam Score Predictor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ASSETS, 'score_05_shap_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP summary for score predictor saved!')

In [ ]:
# ── CELL 11 ── Predict for a Single Student (Demo) ────────────
# This is what Streamlit dashboard will use
print('=== DEMO: Predict score for a single student ===')

# Take one student from test set
student = X_test_df.iloc[[0]]
actual  = y_test.values[0]

predicted = xgb_reg.predict(student)[0]

print(f'Actual grade    : {actual:.2f} / 20')
print(f'Predicted grade : {predicted:.2f} / 20')
print(f'Error           : {abs(actual - predicted):.2f} marks')

# Grade band
def grade_band(score):
    if score >= 16: return '🟢 Excellent'
    elif score >= 13: return '🔵 Good'
    elif score >= 10: return '🟡 Average'
    else: return '🔴 At Risk'

print(f'\nActual band    : {grade_band(actual)}')
print(f'Predicted band : {grade_band(predicted)}')

In [ ]:
# ── CELL 12 ── Final Summary ───────────────────────────────────
print('=' * 55)
print('MODULE 3 — EXAM SCORE PREDICTOR COMPLETE ✅')
print('=' * 55)
print(f"{'Metric':<10} {'Linear Reg':>15} {'XGBoost':>15}")
print('-' * 42)
print(f"{'RMSE':<10} {lr_rmse:>15.4f} {xgb_rmse:>15.4f}")
print(f"{'MAE':<10} {lr_mae:>15.4f} {xgb_mae:>15.4f}")
print(f"{'R²':<10} {lr_r2:>15.4f} {xgb_r2:>15.4f}")
print('-' * 42)
print(f'\n🏆 XGBoost is better model')
print(f'\n📁 Saved:')
print(f'   models/score_predictor.pkl')
print(f'   models/score_scaler.pkl')
print(f'   models/score_features.pkl')
print(f'\n📊 Charts saved to assets/:')
print(f'   score_01_actual_vs_predicted.png')
print(f'   score_02_residuals.png')
print(f'   score_03_model_comparison.png')
print(f'   score_04_feature_importance.png')
print(f'   score_05_shap_summary.png')
print(f'\n🚀 Next: 06_study_plan.ipynb')